# Blood Cell Type Classification with a CNN

This notebook trains a small convolutional neural network (CNN) to classify white blood cell images into four types: eosinophil, lymphocyte, monocyte and neutrophil. It uses data augmentation, early stopping and a validation split, then evaluates on a held-out test set with accuracy, a classification report and a confusion matrix. Finally it predicts the class of single images and flags low-confidence predictions as "unknown".

**How to run:** Open in Google Colab, choose *Runtime → Change runtime type → GPU*, then run all cells. The dataset is downloaded from Kaggle with `kagglehub` (you may need to sign in to Kaggle / add Kaggle credentials in Colab the first time).

_Rewritten from my original Colab notebook: `Blod_cell_classified_9_pm.ipynb`._

## 1. Imports and seeds

In [ ]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices("GPU"))

## 2. Download the dataset

We use the public Kaggle dataset [Blood Cell Images](https://www.kaggle.com/datasets/paultimothymooney/blood-cells) (`paultimothymooney/blood-cells`). It contains augmented images already split into `TRAIN` and `TEST` folders, with one sub-folder per cell type.

In [ ]:
import kagglehub

dataset_path = kagglehub.dataset_download("paultimothymooney/blood-cells")
print("Downloaded to:", dataset_path)

# Find the TRAIN and TEST folders inside the download
train_dir, test_dir = None, None
for root, dirs, files in os.walk(dataset_path):
    if "TRAIN" in dirs and "TEST" in dirs:
        train_dir = os.path.join(root, "TRAIN")
        test_dir = os.path.join(root, "TEST")
        break
print("Train folder:", train_dir)
print("Test folder:", test_dir)

## 3. Load the images

`image_dataset_from_directory` reads the images and uses the folder names as labels. 20% of the training images are kept aside for validation.

In [ ]:
IMG_SIZE = (128, 128)
BATCH_SIZE = 32

train_ds = keras.utils.image_dataset_from_directory(
    train_dir, validation_split=0.2, subset="training", seed=SEED,
    image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode="categorical")
val_ds = keras.utils.image_dataset_from_directory(
    train_dir, validation_split=0.2, subset="validation", seed=SEED,
    image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode="categorical")
test_ds = keras.utils.image_dataset_from_directory(
    test_dir, image_size=IMG_SIZE, batch_size=BATCH_SIZE,
    label_mode="categorical", shuffle=False)  # keep order so labels line up with predictions

class_names = train_ds.class_names
print("Classes:", class_names)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)

## 4. Look at some sample images

In [ ]:
plt.figure(figsize=(10, 10))
for images, labels in train_ds.take(1):
    for i in range(9):
        plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(class_names[np.argmax(labels[i])])
        plt.axis("off")
plt.show()

## 5. Build the CNN

The model applies light random augmentation (only active during training), rescales pixels to 0–1, then uses three convolution blocks and a small dense head.

In [ ]:
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1),
])

model = keras.Sequential([
    keras.Input(shape=IMG_SIZE + (3,)),
    data_augmentation,
    layers.Rescaling(1.0 / 255),
    layers.Conv2D(32, 3, activation="relu", padding="same"),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),
    layers.Conv2D(64, 3, activation="relu", padding="same"),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),
    layers.Conv2D(128, 3, activation="relu", padding="same"),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),
    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(len(class_names), activation="softmax"),
])

model.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss="categorical_crossentropy",
              metrics=["accuracy"])
model.summary()

## 6. Train

Early stopping halts training when validation loss stops improving and restores the best weights. The best model is also saved to disk.

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
    keras.callbacks.ModelCheckpoint("best_blood_cell_classifier.keras", monitor="val_loss", save_best_only=True),
]

history = model.fit(train_ds, validation_data=val_ds, epochs=30, callbacks=callbacks)

In [ ]:
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history["accuracy"], label="train")
plt.plot(history.history["val_accuracy"], label="validation")
plt.title("Accuracy"); plt.xlabel("epoch"); plt.legend()
plt.subplot(1, 2, 2)
plt.plot(history.history["loss"], label="train")
plt.plot(history.history["val_loss"], label="validation")
plt.title("Loss"); plt.xlabel("epoch"); plt.legend()
plt.show()

## 7. Evaluate on the test set

In [ ]:
test_loss, test_acc = model.evaluate(test_ds)
print(f"Test accuracy: {test_acc:.4f}")

# Predictions and true labels (test_ds is not shuffled, so the order matches)
y_prob = model.predict(test_ds)
y_pred = np.argmax(y_prob, axis=1)
y_true = np.concatenate([np.argmax(labels.numpy(), axis=1) for _, labels in test_ds])

print(classification_report(y_true, y_pred, target_names=class_names))

cm = confusion_matrix(y_true, y_pred)
ConfusionMatrixDisplay(cm, display_labels=class_names).plot(cmap="Blues", xticks_rotation=45)
plt.title("Confusion matrix (test set)")
plt.show()

## 8. Predict a single image

If the model's highest probability is below a threshold, we report the image as "unknown" instead of forcing a class.

In [ ]:
CONFIDENCE_THRESHOLD = 0.7

def predict_image(path):
    img = keras.utils.load_img(path, target_size=IMG_SIZE)
    arr = keras.utils.img_to_array(img)
    arr = np.expand_dims(arr, axis=0)  # the model rescales pixels itself
    probs = model.predict(arr, verbose=0)[0]
    best = int(np.argmax(probs))
    label = class_names[best] if probs[best] >= CONFIDENCE_THRESHOLD else "Unknown / low confidence"
    plt.imshow(img); plt.axis("off")
    plt.title(f"{label} ({probs[best]:.2f})")
    plt.show()
    return label, float(probs[best])

# Try one image from each class in the test folder
for name in class_names:
    folder = os.path.join(test_dir, name)
    first_file = sorted(os.listdir(folder))[0]
    print("True:", name, "->", predict_image(os.path.join(folder, first_file)))